# Week 7 — Web Defense & Detection: blocklists lose, boundaries win

**Lesson plan:** [`../weeks/week-07.md`](../weeks/week-07.md)
**Lab:** [`../labs/vuln-web/README.md`](../labs/vuln-web/README.md) (the week-6 app)
**Reading:** OWASP WAF / input-validation cheat sheets + a short piece on WAF
bypasses.

Week 6 *attacked* the web app. This week defends it — and the central result is
uncomfortable: the popular defense (a **WAF blocklist**) is bypassable and noisy,
while the boring defense (**parameterization / positive validation**) actually
guarantees the bug can't happen. Same split as the whole course: raises cost vs.
gives a guarantee.

> ### The one idea
> A blocklist enumerates *badness* — and you can never enumerate all of it, so
> attackers bypass it while legitimate users trip it (false positives). A **positive
> model** — treat input as data (parameterized queries), or allow only known-good —
> makes the attack *structurally impossible*. Defend at the boundary, not with a
> pattern list.

## 1 · A WAF blocklist — the popular first line

A Web Application Firewall inspects requests and blocks ones matching
attack signatures. Here's a small ruleset in the spirit of ModSecurity's Core Rule
Set: regexes for the SQLi/XSS patterns from week 6.

In [ ]:
import re

WAF_RULES = [
    (re.compile(r"(?i)\bor\b\s+\d+\s*=\s*\d+"), "SQLi numeric tautology"),
    (re.compile(r"(?i)union[\s/*]+select"),           "SQLi UNION"),
    (re.compile(r"(?i)<\s*script"),                   "XSS <script>"),
    (re.compile(r"(?i)drop[\s/*]+table"),             "SQLi DROP TABLE"),
]

def waf_block(payload):
    """Return the rule name that fired, or None if the request passes."""
    for rx, name in WAF_RULES:
        if rx.search(payload):
            return name
    return None

# The week-6 attacks, straight out of the box.
attacks = [
    "admin' OR 1=1--",
    "x' UNION SELECT username, password FROM users--",
    "<script>alert(1)</script>",
    "'; DROP TABLE users--",
]
print("WAF vs. the naive week-6 attacks:")
for a in attacks:
    r = waf_block(a)
    print(f"  {'BLOCKED' if r else 'PASSED ':8} {a[:46]}")
print(f"\ncoverage on known attacks: {sum(bool(waf_block(a)) for a in attacks)}/{len(attacks)} — looks great!")

## 2 · ⚠️ ...but blocklists lose (the week-6 lesson, now on defense)

A blocklist blocks the payloads you thought of. The attacker's job is to write one
you didn't. Same intent, different surface form — and the regex misses it.

In [ ]:
bypasses = [
    ("admin' OR 'a'='a'--",                 "non-numeric tautology (rule wants \\d=\\d)"),
    ("x' UnIoN SeLeCt u,p FROM users--",     "case + spacing variant"),
    ("<img src=x onerror=alert(1)>",         "XSS with no <script tag"),
    ("'; DR\u200bOP TABLE users--",          "zero-width char splits DROP"),
]
print("bypasses — same attack, evades the blocklist:")
through = 0
for payload, why in bypasses:
    r = waf_block(payload)
    if not r:
        through += 1
    print(f"  {'BLOCKED' if r else 'PASSED ':8} {payload[:34]:36} ({why})")
print(f"\n{through}/{len(bypasses)} bypasses got through.")
print("""
This is week 6's 'blocklists lose' from the defender's chair. Every bypass is a
five-minute mutation. You can add a rule for each — and the attacker adds a variant
for each rule. The blocklist can never be complete, because the space of 'bad' is
open-ended.""")

## 3 · ⚠️ And blocklists have a false-positive cost (axis 5)

The other blade: a rule broad enough to catch variants also catches **legitimate
traffic**. Every false positive is a real user blocked, a support ticket, an
analyst-hour — and a WAF that blocks real users gets **turned off**, at which point
its coverage is zero.

In [ ]:
benign = [
    "and 1=1 is a tautology i learned in logic class",
    "i want to select the union option on the benefits form",
    "drop table tennis from the list of activities please",
    "reserve the drop table at the makerspace for friday",
    "search for wireless headphones under fifty dollars",
    "my password reset link expired, can you resend it",
]
fps = [b for b in benign if waf_block(b)]
print(f"benign requests blocked: {len(fps)}/{len(benign)} — the false-positive cost\n")
for b in benign:
    r = waf_block(b)
    tag = f"FALSE POSITIVE ({r})" if r else "ok"
    print(f"  {tag:28} {b[:50]}")
print("""
Two of six legitimate requests blocked. Tune the rules looser to cut false
positives and the bypasses multiply; tune them tighter and real users suffer. That
tradeoff has no good setting — because you're enumerating badness. Axis 4 (bypass)
and axis 5 (false positives) both fail at once.""")

## 4 · The positive model — parameterization makes the bug impossible

Now the boring defense that actually works. A parameterized query sends the SQL
*structure* and the *data* to the database **separately**, so user input is only
ever a value — it can never become code. No blocklist, no signatures, nothing to
bypass, because there is no code path from input to query structure.

In [ ]:
# Model a parameterized login: the input is compared as a literal value. There is
# NO string concatenation into a query, so no input can alter the query's meaning.
USER_DB = {"admin": "s3cr3t", "alice": "hunter2"}

def parameterized_login(username, password):
    # username/password are DATA. They index a dict; they cannot become SQL.
    return USER_DB.get(username) == password

print("does ANY injection string change behavior under parameterization?\n")
all_payloads = attacks + [p for p, _ in bypasses]
bypassed = 0
for p in all_payloads:
    got_in = parameterized_login(p, "wrong")
    if got_in:
        bypassed += 1
    print(f"  {p[:34]:36} -> {'LOGGED IN (!)' if got_in else 'denied — input is data, not code'}")
print(f"\ninjections that succeeded against parameterization: {bypassed}/{len(all_payloads)}")
assert bypassed == 0
print("""
Zero. Not 'blocked by a rule' — STRUCTURALLY unable to inject, because the input
never reaches a code context. That's a guarantee you can state (axis 2): 'user
input cannot alter query structure, for any input.' The WAF could never say that.""")

## 5 · So why do WAFs exist at all?

Not as the primary control — as **defense in depth** and, crucially, **detection**.
A WAF is worth running because:

- It's a **fast-deployable stopgap** for a known CVE while you fix the code (virtual
  patching). Hours, not the weeks a code fix might take.
- It **logs attempts**, giving you visibility: someone is probing, from where, with
  what. That's the detection half of this week.
- It raises the attacker's cost, which is real value — as long as you never mistake
  it for a guarantee.

The failure is deploying a WAF *instead of* fixing the code. It's a smoke detector,
not a fireproof wall — and you still don't store the gasoline next to the stove.

In [ ]:
# Detection: even bypassed attempts leave a trail. Log every request, flag the
# suspicious ones for review, and measure the analyst load (week-13's lens).
import collections

traffic = attacks + [p for p, _ in bypasses] + benign
log = []
for req in traffic:
    rule = waf_block(req)
    log.append({"request": req[:40], "blocked": bool(rule), "rule": rule})

blocked = sum(r["blocked"] for r in log)
# A real attack that BYPASSED the WAF is the dangerous case — invisible to blocking,
# but a behavior-based detector (e.g. "quote followed by SQL keyword") might still flag it.
def anomaly(req):
    return bool(re.search(r"['\";].*(select|drop|or|union|onerror)", req, re.I))

flagged_for_review = [r for r in log if not r["blocked"] and anomaly(r["request"])]
print(f"requests: {len(log)}, hard-blocked: {blocked}, "
      f"flagged for human review (bypassed but anomalous): {len(flagged_for_review)}")
for r in flagged_for_review:
    print(f"  REVIEW: {r['request']}")
print("""
Blocking is binary and bypassable; DETECTION widens the net for a human. But every
flag is analyst time (week 13's automation paradox), so detection is a
precision/recall tradeoff too — not a free win.""")

## 6 · The scorecard view — negative vs. positive security

| Control | Model | Guarantee (axis 2) | Bypass (axis 4) | FP cost (axis 5) |
|---|---|---|---|---|
| WAF blocklist | negative (enumerate bad) | **none** | yes — trivial variants | high — blocks real users |
| Parameterized queries | positive (input is data) | **yes — input can't become code** | none | zero |
| Allowlist validation | positive (enumerate good) | strong, for constrained inputs | hard | low if the allowlist fits |
| WAF as detection | — | none, but **visibility** | n/a | analyst load (wk13) |

> **Enumerating badness is a losing game; enumerating or structurally enforcing
> goodness wins.** The WAF isn't useless — it's a stopgap and a sensor — but the
> control with the guarantee is the boring one: fix the code so the input can't
> reach a code context. Defend at the boundary, not with a pattern list.

## 7 · Your studio deliverable

In `week07/`, against your own `vuln-web` (or DVWA):

1. **Deploy a WAF filter** — write a small blocklist; confirm it blocks the week-6
   attacks. Report coverage on known attacks.
2. **Bypass it (axis 4)** — find ≥ 3 payloads with the same intent that evade your
   rules. This is required; a defense you haven't bypassed is unevaluated.
3. **Measure false positives (axis 5)** — run benign traffic through; report the FP
   rate and name a legitimate request your rules blocked.
4. **Fix it properly** — parameterize the query (or add allowlist validation) and
   show the same injections now fail structurally. State the guarantee.
5. **Detection** — log all attempts; propose an anomaly rule that flags bypassed
   attacks for review, and note its analyst-load cost.
6. **Control Scorecard** — WAF vs. parameterization across all axes. The headline:
   which one gives a *guarantee*, and why the other is still worth running.